**Google Colab setup**

In [1]:
!pip -q install pyspark
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
spark = SparkSession.builder.appName("StreamingSet4").getOrCreate()
files.upload() # Select all six CSV files
def read_csv(name):
  return spark.read.csv(name, header=True, inferSchema=True)
sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")
print("Main sessions:", sessions_df.count())
sessions_df.printSchema()
sessions_df.show(5, truncate=False)

Saving content_catalog_120.csv to content_catalog_120.csv
Saving device_master_8.csv to device_master_8.csv
Saving subscriber_master_2400.csv to subscriber_master_2400.csv
Saving subscription_plans_4.csv to subscription_plans_4.csv
Saving watch_sessions_80000.csv to watch_sessions_80000.csv
Saving watch_sessions_extra_10000.csv to watch_sessions_extra_10000.csv
Main sessions: 80000
root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

+----------+-------------+----------+---------+-------------------+----

**Part A - DataFrame operations**

In [4]:
#1
from pyspark.sql.types import NumericType, StringType, TimestampType
sessions_df.printSchema()
sessions_df.count()
sessions_df.show(10)
numeric_field = [field.name for field in sessions_df.schema.fields if isinstance(field.dataType, NumericType)]
string_field = [field.name for field in sessions_df.schema.fields if isinstance(field.dataType, StringType)]
timestamp_field = [field.name for field in sessions_df.schema.fields if isinstance(field.dataType, TimestampType)]
print("Numeric fields:", numeric_field)
print("String fields:", string_field)
print("Timestamp fields:", timestamp_field)

root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--

In [5]:
#2
sessions_df.select(
    'session_id',
    'subscriber_id',
    'content_id',
    'watch_minutes',
    'session_status'
).filter(
    (F.col('session_status') == 'Completed') &
    (F.col('watch_minutes') > 60)
).show()


+----------+-------------+----------+-------------+--------------+
|session_id|subscriber_id|content_id|watch_minutes|session_status|
+----------+-------------+----------+-------------+--------------+
|WS00000049|     SUB01859|    CT0075|          125|     Completed|
|WS00000179|     SUB00935|    CT0054|           62|     Completed|
|WS00000184|     SUB00072|    CT0119|           91|     Completed|
|WS00000222|     SUB01603|    CT0103|          116|     Completed|
|WS00000233|     SUB01233|    CT0027|          158|     Completed|
|WS00000244|     SUB00985|    CT0048|          140|     Completed|
|WS00000249|     SUB00681|    CT0100|          120|     Completed|
|WS00000257|     SUB02115|    CT0120|           68|     Completed|
|WS00000292|     SUB01068|    CT0048|          141|     Completed|
|WS00000326|     SUB00368|    CT0096|           97|     Completed|
|WS00000415|     SUB01758|    CT0105|          150|     Completed|
|WS00000419|     SUB00367|    CT0036|           99|     Comple

In [32]:
#3
sessions_df1 = sessions_df.withColumn(
    'watch_hours',
    F.col('watch_minutes') / 60
).withColumn(
    'High_engagement_flag',
    F.when(F.col('completion_pct').isNull(), None)
    .when(F.col('completion_pct') >= 75, 'Yes')
    .otherwise('No')
)
sessions_df1.show()

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-------------------+--------------------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|        watch_hours|High_engagement_flag|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-------------------+--------------------+
|WS00000001|     SUB00157|    CT0097|     DV02|2026-04-26 18:59:00|            6|          11.8|             29.0|            HD|     Abandoned|        72.4|       NULL|                0.1|                  No|
|WS00000002|     SUB01430|    CT0031|     DV04|2026-07-30 06:26:00|           31|          56.4|             31.0|            HD|   Interrupted|       422.7

In [7]:
#4
sessions_df.sort(
    F.col('data_used_mb'),ascending= False
).limit(20).show()

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00051169|     SUB00155|    CT0069|     DV08|2026-07-20 00:11:00|          167|          97.7|              9.0|            4K|     Completed|      8881.8|        3.0|
|WS00074513|     SUB00684|    CT0056|     DV05|2026-08-09 21:53:00|          157|          96.9|              8.0|            4K|     Completed|      8408.4|       NULL|
|WS00039201|     SUB01664|    CT0027|     DV08|2026-05-16 06:35:00|          161|          97.0|              9.0|            4K|     Completed|      

**Part B - Data quality and conditional transformations**

In [8]:
#5
sessions_df.select(
    F.sum(F.when(F.col('subscriber_id').isNull(), 1).otherwise(0)).alias('subscriber_id_null_count'),
    F.sum(F.when(F.col('device_id').isNull(), 1).otherwise(0)).alias('device_id_null_count'),
    F.sum(F.when(F.col('completion_pct').isNull(), 1).otherwise(0)).alias('completion_pct_null_count'),
    F.sum(F.when(F.col('buffering_seconds').isNull(), 1).otherwise(0)).alias('buffering_seconds_null_count'),
    F.sum(F.when(F.col('user_rating').isNull(), 1).otherwise(0)).alias('user_rating_null')
).show()

+------------------------+--------------------+-------------------------+----------------------------+----------------+
|subscriber_id_null_count|device_id_null_count|completion_pct_null_count|buffering_seconds_null_count|user_rating_null|
+------------------------+--------------------+-------------------------+----------------------------+----------------+
|                     951|                2003|                     3513|                        4424|           77605|
+------------------------+--------------------+-------------------------+----------------------------+----------------+



In [9]:
#6
new_df = sessions_df.na.fill(
    0,
    subset = ['buffering_seconds']
)
new_df.show()

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00000001|     SUB00157|    CT0097|     DV02|2026-04-26 18:59:00|            6|          11.8|             29.0|            HD|     Abandoned|        72.4|       NULL|
|WS00000002|     SUB01430|    CT0031|     DV04|2026-07-30 06:26:00|           31|          56.4|             31.0|            HD|   Interrupted|       422.7|       NULL|
|WS00000003|     SUB01892|    CT0044|     DV08|2026-02-22 20:55:00|           29|          63.0|             26.0|            HD|   Interrupted|      

In [10]:
#7
current_count = sessions_df.count()
new_df = sessions_df.dropna(subset=['device_id'])
new_count = new_df.count()
print("Number of rows before dropping:",current_count)
print("Number of rows after dropping:",new_count)

Number of rows before dropping: 80000
Number of rows after dropping: 77997


In [11]:
#8
new_df = sessions_df.withColumn(
    'completion_level',
    F.when(F.col('completion_pct') < 30, 'Low')
    .when((F.col('completion_pct') >= 30) & (F.col('completion_pct') < 90), 'Medium')
    .when(F.col('completion_pct') >= 90, 'High')
    .otherwise('Unknown')
)
new_df.show()

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+----------------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|completion_level|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+----------------+
|WS00000001|     SUB00157|    CT0097|     DV02|2026-04-26 18:59:00|            6|          11.8|             29.0|            HD|     Abandoned|        72.4|       NULL|             Low|
|WS00000002|     SUB01430|    CT0031|     DV04|2026-07-30 06:26:00|           31|          56.4|             31.0|            HD|   Interrupted|       422.7|       NULL|          Medium|
|WS00000003|     SUB01892|    CT0044|     DV08|2026-02-22 20:55:0

**Part C - Joins and aggregations**

In [12]:
#9
session_content = sessions_df.join(
    content_df,
    on = 'content_id',
    how = 'inner'
)
session_content.groupBy('genre').agg(
    F.count('*').alias('session_count'),
    F.sum(F.col('watch_minutes')).alias('total_watch_minutes')
).show()


+-----------+-------------+-------------------+
|      genre|session_count|total_watch_minutes|
+-----------+-------------+-------------------+
|   Thriller|         5972|             248430|
|  Adventure|        15561|             839700|
|     Sports|         9967|             383044|
|      Drama|        10733|             519872|
|Documentary|         7432|             256626|
|  Animation|        11886|             490335|
|     Comedy|         9105|             452563|
|     Sci-Fi|         9344|             433946|
+-----------+-------------+-------------------+



In [13]:
#10
session_device = sessions_df.join(
    devices_df,
    on = 'device_id',
    how = 'inner'
)
session_device.groupBy('device_type').agg(
    F.avg(F.col('buffering_seconds')).alias('avg_buffering_seconds'),
    F.avg(F.col('watch_minutes')).alias('avg_watch_minutes')
).show()



+---------------+---------------------+------------------+
|    device_type|avg_buffering_seconds| avg_watch_minutes|
+---------------+---------------------+------------------+
|         iPhone|   35.761106907361366|45.226752503576535|
|Streaming Stick|   36.320860309448044| 45.08570541351827|
|Desktop Browser|    36.02889125799574|45.571818733562615|
|  Android Phone|   35.676486693680864|  45.0885616509385|
| Laptop Browser|   36.930938301907844| 45.84465713083719|
|         Tablet|    35.23614616481949| 45.38943894389439|
|       Smart TV|    36.67392739273927| 45.33776623376623|
|   Game Console|    35.51243726816496| 45.04469158686026|
+---------------+---------------------+------------------+



In [14]:
#11
session_subscriber_plans = sessions_df.join(
    subscribers_df,
    on = 'subscriber_id',
    how = 'inner'
).join(
    plans_df,
    on = 'plan_id',
    how = 'inner'
)
session_subscriber_plans.filter(
    F.col('subscriber_id').isNotNull()
).groupBy('plan_name').agg(
    F.count('*').alias('total_sessions'),
    F.sum(F.when(F.col('session_status') == 'Completed', 1).otherwise(0)).alias('completed_sessions')
).withColumn(
    'completion_rate',
    F.round(F.col('completed_sessions') / F.col('total_sessions') * 100,2)
)
session_subscriber_plans.show()

+-------+-------------+----------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-----------+-------+---------+-----------------+---------+---------------+----------------------+-----------------+
|plan_id|subscriber_id|session_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|signup_date| region|age_group|subscriber_status|plan_name|monthly_fee_inr|max_concurrent_streams|max_video_quality|
+-------+-------------+----------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-----------+-------+---------+-----------------+---------+---------------+----------------------+-----------------+
|   PL02|     SUB00157|WS00000001|    CT0097|     DV02|2026-04-26 18:59:00|            6|          11.8|             29.0

In [15]:
#12
subscribers_plans = subscribers_df.join(
    plans_df,
    on = 'plan_id',
    how = 'inner'
)
subscribers_plans.filter(
    F.col('subscriber_status') == 'Active'
).dropDuplicates(
    ['subscriber_id']
).groupBy(
    'plan_name'
).agg(
    F.sum(F.col('monthly_fee_inr')).alias('total_monthly_fee')
).show()

+---------+-----------------+
|plan_name|total_monthly_fee|
+---------+-----------------+
|  Premium|           211574|
|    Basic|           157368|
|   Mobile|            66007|
| Standard|           240996|
+---------+-----------------+



In [16]:
#13
active_subscribers = sessions_df.filter(
    F.col('subscriber_id').isNotNull()
)
active_devices = sessions_df.filter(
    F.col('device_id').isNotNull()
)
unmatched_subscribers = active_subscribers.join(
    subscribers_df,
    on = 'subscriber_id',
    how = 'left_anti'
)
unmatched_devices = active_devices.join(
    devices_df,
    on = 'device_id',
    how = 'left_anti'
)
print("Unmatched subscribers count:", unmatched_subscribers.select('subscriber_id').count())
print("Unmatched devices count:", unmatched_devices.select('device_id').count())

Unmatched subscribers count: 571
Unmatched devices count: 481


**Part D - Date and window functions**

In [33]:
#14
from pyspark.sql.functions import year, month, to_timestamp
sessions_df = sessions_df.withColumn(
    'session_start',
    to_timestamp(F.col('session_start'))
)
monthly_metrics = sessions_df.filter(
    year(F.col('session_start')) == 2026
).groupBy(
    month(F.col('session_start')).alias('month')
).agg(
    F.count('*').alias('session_count'),
    F.sum(F.col('watch_minutes')).alias('total_watch_minutes')
).orderBy('month')
monthly_metrics.show()

+-----+-------------+-------------------+
|month|session_count|total_watch_minutes|
+-----+-------------+-------------------+
|    1|         9070|             407215|
|    2|         8218|             372726|
|    3|         9019|             403971|
|    4|         8817|             399432|
|    5|         9006|             407386|
|    6|         8776|             404203|
|    7|         9204|             416104|
|    8|         8996|             407374|
|    9|         8894|             406105|
+-----+-------------+-------------------+



In [18]:
#15
from pyspark.sql.functions import date_diff
session_subscriber = sessions_df.join(
    subscribers_df,
    on = 'subscriber_id',
    how = 'inner'
)
session_subscriber.withColumn(
    'days_between_signup',
    date_diff(F.col('session_start'), F.col('signup_date'))
).orderBy(F.col('session_start').asc()).show(10)

+-------------+----------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-----------+------+-------+---------+-----------------+-------------------+
|subscriber_id|session_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|signup_date|region|plan_id|age_group|subscriber_status|days_between_signup|
+-------------+----------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-----------+------+-------+---------+-----------------+-------------------+
|     SUB01707|WS00035235|    CT0103|     DV07|2026-01-01 00:00:00|           60|          51.3|             11.0|            HD|   Interrupted|       819.7|       NULL| 2024-02-13|  East|   PL04|    18-24|           Active|                688|
|     SUB02206|WS000

In [34]:
#16
from pyspark.sql.functions import day
sessions_df2 = sessions_df.withColumn(
  'day_type',
  F.when(F.dayofweek('session_start').isin(1, 7),'Weekend'
    ).otherwise('Weekday')
)
sessions_df2.groupBy('day_type').agg(
    F.avg(F.col('watch_minutes')).alias('avg_watch_minutes')
).show()

+--------+------------------+
|day_type| avg_watch_minutes|
+--------+------------------+
| Weekday| 45.26460823373174|
| Weekend|45.411601967328295|
+--------+------------------+



In [20]:
#17
from pyspark.sql.functions import dense_rank

agg_df = session_content.groupBy('genre','content_title').agg(
    F.sum(F.col('watch_minutes')).alias('total_watch_minutes')
)

window_spec = Window.partitionBy('genre').orderBy(F.col('total_watch_minutes').desc())

agg_df.withColumn(
    'rank',
    dense_rank().over(window_spec)
).filter(
    F.col('rank') <= 3
).show()

+-----------+----------------+-------------------+----+
|      genre|   content_title|total_watch_minutes|rank|
+-----------+----------------+-------------------+----+
|  Adventure|Infinite Horizon|              66395|   1|
|  Adventure|    Lost Journey|              61823|   2|
|  Adventure|    Solar Memory|              60540|   3|
|  Animation|      Last River|              51398|   1|
|  Animation| Emerald Circuit|              51310|   2|
|  Animation|   Golden Puzzle|              50515|   3|
|     Comedy|    Blue Horizon|              61428|   1|
|     Comedy|   Bright Valley|              60921|   2|
|     Comedy|   Silver Valley|              60422|   3|
|Documentary|   Fading Signal|              47085|   1|
|Documentary|  Emerald Valley|              40819|   2|
|Documentary|   Solar Horizon|              36152|   3|
|      Drama|  Crimson Summit|              65724|   1|
|      Drama|  Rising Archive|              57301|   2|
|      Drama|     Wild Harbor|              4547

In [21]:
#18
from pyspark.sql.functions import row_number

non_null_subscriber_id = sessions_df.filter(
    F.col('subscriber_id').isNotNull()
)
window_spec = Window.partitionBy('subscriber_id').orderBy(F.col('session_start').desc())
non_null_subscriber_id.withColumn(
    'session_number',
    row_number().over(window_spec)
).filter(
    F.col('session_number') == 1
).show()


+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+--------------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|session_number|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+--------------+
|WS00020822|     SUB00001|    CT0073|     DV03|2026-09-11 19:19:00|           34|          89.5|            145.0|            HD|   Interrupted|       418.3|       NULL|             1|
|WS00032744|     SUB00002|    CT0076|     DV05|2026-09-02 05:41:00|           45|          73.8|            219.0|       Full HD|   Interrupted|      1184.1|       NULL|             1|
|WS00006110|     SUB00003|    CT0004|     DV03|2026-09-30 03:13:00|        

In [22]:
#19
from pyspark.sql.functions import lag

window_spec = Window.partitionBy('subscriber_id').orderBy(
  'session_start', 'session_id'
)
filtered_sessions_df = sessions_df.filter(F.col('subscriber_id').isNotNull())

filtered_sessions_df = filtered_sessions_df.withColumn(
    'previous_watch_minutes',
    F.lag(F.col('watch_minutes')).over(window_spec)
)
filtered_sessions_df.select(
    'subscriber_id',
    'session_id',
    'watch_minutes',
    'previous_watch_minutes'
).show()


+-------------+----------+-------------+----------------------+
|subscriber_id|session_id|watch_minutes|previous_watch_minutes|
+-------------+----------+-------------+----------------------+
|     SUB00001|WS00064390|           58|                  NULL|
|     SUB00001|WS00045183|           78|                    58|
|     SUB00001|WS00028484|            4|                    78|
|     SUB00001|WS00043555|          108|                     4|
|     SUB00001|WS00041862|           79|                   108|
|     SUB00001|WS00059616|           18|                    79|
|     SUB00001|WS00042345|           44|                    18|
|     SUB00001|WS00009166|           91|                    44|
|     SUB00001|WS00003744|           27|                    91|
|     SUB00001|WS00029676|           63|                    27|
|     SUB00001|WS00043178|          113|                    63|
|     SUB00001|WS00014354|           28|                   113|
|     SUB00001|WS00008269|            6|

**Part E - Union, lazy execution, caching and partitioning**

In [23]:
#20
combine = sessions_df.unionByName(
    extra_df,
    allowMissingColumns=True
)
print("number of rows:",combine.count())
combine = combine.dropDuplicates(subset=['session_id'])
print("Removing duplicates:",combine.count())

number of rows: 90000
Removing duplicates: 89400


In [24]:
#21
filter = sessions_df.filter(
    F.col('session_status') == 'Completed'
)
join = filter.join(
    subscribers_df,
    on = 'subscriber_id',
    how = 'inner'
)

add_column = join.withColumn(
    'is_active',
    F.when(F.col('subscriber_status') == 'Active', 1).otherwise(0)
)


group = add_column.groupBy('region').agg(
    F.sum(F.col('is_active')).alias('total_active_subscribers')
)

ordering = group.orderBy(F.col('total_active_subscribers').desc())

ordering.explain()

ordering.show()



== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_active_subscribers#1318L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_active_subscribers#1318L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2263]
      +- HashAggregate(keys=[region#77], functions=[sum(is_active#1317)])
         +- Exchange hashpartitioning(region#77, 200), ENSURE_REQUIREMENTS, [plan_id=2260]
            +- HashAggregate(keys=[region#77], functions=[partial_sum(is_active#1317)])
               +- Project [region#77, CASE WHEN (subscriber_status#80 = Active) THEN 1 ELSE 0 END AS is_active#1317]
                  +- BroadcastHashJoin [subscriber_id#18], [subscriber_id#75], Inner, BuildRight, false
                     :- Project [subscriber_id#18]
                     :  +- Filter ((isnotnull(session_status#26) AND (session_status#26 = Completed)) AND isnotnull(subscriber_id#18))
                     :     +- FileScan csv [subscriber_id#18,session_status#26] Batched: false,

In [26]:
from pickleshare import stat
from pyspark.sql.functions.builtin import aggregate
#22
import time
completed_session = sessions_df.filter(
    F.col('session_status') == 'Completed'
)

completed_session.cache()

start_time = time.time()
completed_session.count()
aggregate = completed_session.groupBy('subscriber_id').agg(
    F.sum(F.col('watch_minutes')).alias('total_watch_minutes')
)
aggregate.show()
end_time = time.time()
print("Time taken:", end_time - start_time, "seconds")

start_time = time.time()
completed_session.count()
aggregate = completed_session.groupBy('subscriber_id').agg(
    F.sum(F.col('watch_minutes')).alias('total_watch_minutes')
)
aggregate.show()
end_time = time.time()
print("Time taken:", end_time - start_time, "seconds")

'''cache() is lazy. It doesn't immediately store data in memory.
The first count() trigger execution and populate the cache.
The furthur actions will reuse the cache.'''

+-------------+-------------------+
|subscriber_id|total_watch_minutes|
+-------------+-------------------+
|     SUB01109|                156|
|     SUB00932|                649|
|     SUB00118|                148|
|     SUB02400|                152|
|     SUB00013|                253|
|     SUB01389|                283|
|     SUB01933|                167|
|     SUB02174|                 43|
|     SUB00938|                258|
|     SUB01320|                408|
|     SUB02074|                302|
|     SUB00533|                138|
|     SUB00212|                198|
|     SUB00564|                146|
|     SUB01985|                310|
|     SUB01852|                199|
|     SUB00656|                133|
|     SUB00727|                169|
|     SUB01749|                118|
|     SUB00954|                208|
+-------------+-------------------+
only showing top 20 rows
Time taken: 1.403372049331665 seconds
+-------------+-------------------+
|subscriber_id|total_watch_minutes|
+

"cache() is lazy. It doesn't immediately store data in memory. \nThe first count() trigger execution and populate the cache.\nThe furthur actions will reuse the cache."

In [27]:
#23
stream_quality = sessions_df.filter(
    F.col('stream_quality') == '4K'
)
stream_quality.persist(StorageLevel.MEMORY_AND_DISK)

#actions
stream_quality.count()
stream_quality.show()

stream_quality.unpersist()

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00000005|     SUB01268|    CT0052|     DV08|2026-04-06 04:33:00|           36|          23.4|             31.0|            4K|     Abandoned|      1561.1|       NULL|
|WS00000010|     SUB00947|    CT0063|     DV02|2026-08-09 09:54:00|           21|          65.6|            166.0|            4K|   Interrupted|      1065.3|       NULL|
|WS00000017|     SUB02006|    CT0027|     DV04|2026-01-15 14:05:00|          140|          84.3|             12.0|            4K|   Interrupted|      

DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

In [28]:
#24
from pyspark.sql.functions import spark_partition_id
partition_df8 = sessions_df.repartition(8)
partition_df8.groupBy(spark_partition_id()).agg(
    F.count('*').alias('session_count')
).show()


+--------------------+-------------+
|SPARK_PARTITION_ID()|session_count|
+--------------------+-------------+
|                   0|        10000|
|                   1|         9999|
|                   2|        10001|
|                   3|        10000|
|                   4|        10000|
|                   5|        10000|
|                   6|        10000|
|                   7|        10000|
+--------------------+-------------+



In [29]:
#25
reduce_coalesce = partition_df8.coalesce(2)
reduce_repartition = partition_df8.repartition(2)
print("Coalesce plan: ")
reduce_coalesce.explain()
print("Repartition plan: ")
reduce_repartition.explain()

reduce_repartition.write.mode('overwrite').parquet('output')

Coalesce plan: 
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 2
   +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=2813]
      +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_status#26,data_used_mb#27,user_rating#28] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/watch_sessions_80000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<session_id:string,subscriber_id:string,content_id:string,device_id:string,session_start:ti...


Repartition plan: 
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=2822]
   +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_status#26

**Final capstone - Streaming performance report**

In [37]:
#1
from pyspark.sql.functions import rank
from pyspark.sql.window import Window
union = sessions_df.unionByName(
    extra_df,
    allowMissingColumns=True
)
#deduplication
union.groupBy('session_id').agg(
    F.sum(F.when(F.col('session_id').isNotNull(), 1).otherwise(0)).alias('session_count')
).filter(
    F.col('session_count') > 1
).show()

union = union.dropDuplicates(subset=['session_id'])

#2
cleaned_df = sessions_df.na.fill(
    0,
    subset = ['buffering_seconds']
)

#3
joined = cleaned_df.join(
    subscribers_df,
    on = 'subscriber_id',
    how = 'inner'
).join(
    content_df,
    on = 'content_id',
    how = 'inner'
).join(
    devices_df,
    on = 'device_id',
    how = 'inner'
).join(
    plans_df,
    on = 'plan_id',
    how = 'inner'
)

#4
monthly_report = joined.groupBy(F.month(F.col('session_start')).alias('month'), 'region', 'genre').agg(
    F.count('session_id').alias('session_count'),
    F.sum(F.col('watch_minutes')).alias('total_watch_minutes'),
    F.sum(F.col('completion_pct')).alias('total_completion_pct'),
    F.avg(F.col('buffering_seconds')).alias('avg_buffering_seconds'),
    F.sum(F.col('data_used_mb')).alias('total_data_used_mb')
)
monthly_report.show()

#5
group = joined.groupBy('genre','content_title').agg(
    F.sum(F.col('watch_minutes')).alias('total_watch_minutes')
).withColumn(
    'watch_hours',
    F.round(F.col('total_watch_minutes') / 60, 2)
)
window_spec = Window.partitionBy('genre').orderBy(F.col('watch_hours').desc())
top_titles = group.withColumn(
    'rank',
    F.dense_rank().over(window_spec)
).filter(
    F.col('rank') <= 3
)
top_titles.orderBy(
    'genre',
    'rank'
).show(30)

#6

monthly_report.cache()

monthly_report.count()
monthly_report.show()

monthly_report.unpersist()

monthly_report.repartition(3).write.mode(
    'overwrite'
).parquet('output')

+----------+-------------+
|session_id|session_count|
+----------+-------------+
|WS00037768|            2|
|WS00041893|            2|
|WS00036191|            2|
|WS00001914|            2|
|WS00005785|            2|
|WS00007204|            2|
|WS00012187|            2|
|WS00038036|            2|
|WS00051716|            2|
|WS00055255|            2|
|WS00002698|            2|
|WS00011260|            2|
|WS00049034|            2|
|WS00007038|            2|
|WS00050894|            2|
|WS00008270|            2|
|WS00011737|            2|
|WS00028212|            2|
|WS00048522|            2|
|WS00007081|            2|
+----------+-------------+
only showing top 20 rows
+-----+-------+-----------+-------------+-------------------+--------------------+---------------------+------------------+
|month| region|      genre|session_count|total_watch_minutes|total_completion_pct|avg_buffering_seconds|total_data_used_mb|
+-----+-------+-----------+-------------+-------------------+------------------